# Speech Commands — Google Colab Training

Notebook do trenowania modeli z repo **DeepLearningSpeechCommands** na Colab GPU.

Wyniki (modele, historia, wykresy) są zapisywane na **Google Drive natychmiast** po każdym modelu — niezależnie od ewentualnego resetu sesji.

> Runtime → Change runtime type → **T4 GPU** (lub A100 dla Colab Pro)

---
## 1. Sprawdz GPU

In [ ]:
import torch
print(f'PyTorch:  {torch.__version__}')
print(f'CUDA:     {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU:      {torch.cuda.get_device_name(0)}')
    print(f'VRAM:     {p.total_memory / 1e9:.1f} GB')
else:
    print('Brak GPU — Runtime → Change runtime type → GPU')

---
## 2. Zamontuj Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, json, glob

# ―― Ścieżki na Google Drive ――――――――――――――――――――――――――――――――――――――――――
DRIVE_BASE    = '/content/drive/MyDrive/Studia/Projekty/DeepLearning/AudioClassification'
DRIVE_RUNS    = f'{DRIVE_BASE}/runs'
DRIVE_DATASET = f'{DRIVE_BASE}/Dataset'

for d in [DRIVE_BASE, DRIVE_RUNS, DRIVE_DATASET]:
    os.makedirs(d, exist_ok=True)

print(f'Drive base: {DRIVE_BASE}')
print(f'Runs:       {DRIVE_RUNS}')
print(f'Dataset:    {DRIVE_DATASET}')

---
## 3. Klonuj repo z GitHub

Zmien `GITHUB_URL` na adres swojego repozytorium.  
Przy kolejnych sesjach Colab `git pull` pobierze najnowsze zmiany automatycznie.

In [ ]:
import os, sys

REPO_DIR   = '/content/speech_commands'
GITHUB_URL = 'https://github.com/JakKepka/DeepLearningSpeechCommands.git'  # ← zmien!

if os.path.exists(REPO_DIR):
    print(f'Repo istnieje: {REPO_DIR}')
    os.system(f'git -C "{REPO_DIR}" pull --ff-only')
else:
    ret = os.system(f'git clone --depth=1 "{GITHUB_URL}" "{REPO_DIR}"')
    assert ret == 0, 'git clone nie powiodl sie — sprawdz GITHUB_URL'
    print('Repo sklonowane: OK')

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'cwd: {os.getcwd()}')

---
## 4. Instalacja dependencji

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'soundfile', 'transformers', 'scikit-learn', 'seaborn', 'pyyaml', 'tqdm'],
    check=True
)
import torch, torchaudio
print(f'torch {torch.__version__}  |  torchaudio {torchaudio.__version__}')


---
## 5. Dataset Speech Commands (~2.4 GB)

- **Pierwsza sesja:** pobierz i ustaw `SAVE_DATASET_TO_DRIVE = True` → dataset trafia na Drive.
- **Kolejne sesje:** `USE_DRIVE_DATASET_CACHE = True` → ~30 s zamiast kilku minut.

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

DATA_DIR = './data/raw'

# ―― Zmien odpowiednio ―――――――――――――――――――――――――――――――――――――――
USE_DRIVE_DATASET_CACHE = False    # True = kopiuj z Drive
SAVE_DATASET_TO_DRIVE   = False   # True = zapisz na Drive po pobraniu

ds_on_drive = os.path.exists(f'{DRIVE_DATASET}/SpeechCommands')

if USE_DRIVE_DATASET_CACHE and ds_on_drive:
    print('Kopiowanie datasetu z Drive...')
    os.makedirs(DATA_DIR, exist_ok=True)
    src = Path(DRIVE_DATASET) / 'SpeechCommands'
    dst = Path(DATA_DIR) / 'SpeechCommands'
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print('Dataset skopiowany z Drive')
else:
    print('Pobieranie Speech Commands (~2.4 GB)...')

    repo = Path(REPO_DIR)
    env = os.environ.copy()
    env['PYTHONPATH'] = f"{repo}:{env.get('PYTHONPATH', '')}"

    print(f'CWD: {repo}')
    print(f'Python: {sys.executable}')
    print(f'PYTHONPATH(head): {env["PYTHONPATH"].split(":")[0]}')

    # 1) Preferowane uruchomienie jako moduł.
    cmd = [sys.executable, '-m', 'scripts.prepare_data', '--root', DATA_DIR]
    process = subprocess.run(
        cmd,
        cwd=str(repo),
        env=env,
        text=True,
        capture_output=True,
    )

    # 2) Fallback: bezpośrednio plik.
    if process.returncode != 0:
        print('Uruchomienie modułu nieudane, próba fallback do scripts/prepare_data.py...')
        cmd = [sys.executable, str(repo / 'scripts' / 'prepare_data.py'), '--root', DATA_DIR]
        process = subprocess.run(
            cmd,
            cwd=str(repo),
            env=env,
            text=True,
            capture_output=True,
        )

    print('--- Output from prepare_data.py ---')
    if process.stdout:
        print(process.stdout)
    if process.stderr:
        print(process.stderr)
    print('-----------------------------------')

    assert process.returncode == 0, 'prepare_data.py nie powiodlo sie'
    print('Dataset gotowy')

    if SAVE_DATASET_TO_DRIVE and not ds_on_drive:
        print('Zapisuję dataset na Drive...')
        src = Path(DATA_DIR) / 'SpeechCommands'
        dst = Path(DRIVE_DATASET) / 'SpeechCommands'
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'Dataset zapisany: {dst}')

---
## 6. Funkcja automatycznego zapisu na Drive

Wywoływana automatycznie po każdym treningu. Można też uruchomić ręcznie.

In [ ]:
from datetime import datetime

def sync_run_to_drive(experiment: str, seed: int) -> str:
    """Kopiuje wyniki runu na Google Drive. Zwraca sciezke na Drive."""
    pattern = f'outputs/tables/*{experiment}*seed{seed}*history.json'
    files = sorted(glob.glob(pattern))
    if not files:
        print(f'Nie znaleziono: {pattern}')
        return ''

    hist_path = files[-1]
    run_name  = os.path.basename(hist_path).replace('_history.json', '')
    drive_run = f'{DRIVE_RUNS}/{run_name}'
    os.makedirs(f'{drive_run}/checkpoints', exist_ok=True)
    os.makedirs(f'{drive_run}/figures',     exist_ok=True)

    copied = []

    # Historia uczenia
    shutil.copy2(hist_path, f'{drive_run}/{os.path.basename(hist_path)}')
    copied.append(f'history.json')

    # Checkpointy
    for suffix in ('_best.pt', '_last.pt'):
        src = f'outputs/checkpoints/{run_name}{suffix}'
        if os.path.exists(src):
            shutil.copy2(src, f'{drive_run}/checkpoints/{run_name}{suffix}')
            copied.append(os.path.basename(src))

    # Wykresy
    for fig in glob.glob('outputs/figures/*.png'):
        shutil.copy2(fig, f'{drive_run}/figures/{os.path.basename(fig)}')
        copied.append(os.path.basename(fig))

    # Aktualizuj indeks runs/index.json
    idx_path = f'{DRIVE_RUNS}/index.json'
    idx = json.load(open(idx_path)) if os.path.exists(idx_path) else []
    with open(hist_path) as f:
        hdata = json.load(f)
    # unikaj duplikatow
    idx = [r for r in idx if r.get('run_name') != run_name]
    idx.append({
        'run_name':     run_name,
        'experiment':   experiment,
        'seed':         seed,
        'saved_at':     datetime.now().isoformat(timespec='seconds'),
        'best_val_acc': hdata.get('meta', {}).get('best_val_acc'),
        'test_acc':     hdata.get('test', {}).get('accuracy'),
        'total_epochs': hdata.get('meta', {}).get('total_epochs'),
    })
    with open(idx_path, 'w') as f:
        json.dump(idx, f, indent=2)

    print(f'Zapisano na Drive ({drive_run}):')
    for c in copied:
        print(f'  {c}')
    return drive_run

print('sync_run_to_drive() gotowy')

---
## 7. Konfiguracja

| Eksperyment | Model | Param | ~czas/epoka (T4, bs=256) |
|---|---|---|---|
| A1 | CNN Baseline | ~450 k | 3–4 min |
| A2 | KWT-small | ~607 k | ~5 min |
| A3 | KWT-medium | ~5.4 M | ~8 min |
| A4 | AST fine-tune | ~86 M | wymaga Colab Pro |

In [ ]:
import yaml
from src.utils.config import load_experiment_config

# ―― Wybierz eksperyment ―――――――――――――――――――――――――――――――――――――――――
EXPERIMENT = 'A1'   # A1 / A2 / A3 / A4 / C1 / C2 / C3
SEED = 42

# Szybki profil "FAST A1"
AUGMENT_ENABLED = False
NUM_WORKERS = 10
BATCH_SIZE = 512
EARLY_STOPPING_PATIENCE = 4

# Najmocniejszy akcelerator: cache log-mel poza pętlą epok
# none | val_test | all
CACHE_FEATURES = 'all'
CACHE_DTYPE = 'float16'

CONFIG_MAP = {
    'A1': 'configs/experiments/A1_cnn_baseline.yaml',
    'A2': 'configs/experiments/A2_kwt_small.yaml',
    'A3': 'configs/experiments/A3_kwt_medium.yaml',
    'A4': 'configs/experiments/A4_ast.yaml',
    'C1': 'configs/experiments/C1_flat.yaml',
    'C2': 'configs/experiments/C2_flat_rebalanced.yaml',
    'C3': 'configs/experiments/C3_hierarchical.yaml',
}

cfg = load_experiment_config(CONFIG_MAP[EXPERIMENT])
t = cfg.setdefault('train', {})
a = cfg.setdefault('augmentation', {})

a['enabled'] = AUGMENT_ENABLED
t['num_workers'] = NUM_WORKERS
t['prefetch_factor'] = 4
t['early_stopping_patience'] = EARLY_STOPPING_PATIENCE
t['device'] = 'cuda'
t['batch_size'] = BATCH_SIZE
t['tables_dir'] = './outputs/tables'
t['checkpoint_dir'] = './outputs/checkpoints'

t['cache_features'] = CACHE_FEATURES
t['feature_cache_dir'] = './data/feature_cache_colab'
t['cache_dtype'] = CACHE_DTYPE
t['cache_batch_size'] = BATCH_SIZE
t['cache_overwrite'] = False

tmp_cfg = f'/tmp/{EXPERIMENT}_seed{SEED}_colab.yaml'
with open(tmp_cfg, 'w') as f:
    yaml.dump(cfg, f)

print(f'Eksperyment: {EXPERIMENT}  |  seed: {SEED}')
print(
    f'device: {t["device"]} | bs: {t["batch_size"]} | workers: {t["num_workers"]} | '
    f'patience: {t["early_stopping_patience"]} | augment: {a["enabled"]}'
)
print(
    f'cache_features: {t["cache_features"]} | cache_dtype: {t["cache_dtype"]} | '
    f'cache_dir: {t["feature_cache_dir"]}'
)
if t['cache_features'] == 'all' and a['enabled']:
    print('UWAGA: cache all + augment=True zamraża augmentacje. Ustaw augment=False.')

In [ ]:
# Benchmark DataLoader: num_workers 8 / 10 / 12 (A1, augment=False, bs=512)
import copy
import time
import glob
import json
import os
import subprocess

BENCH_WORKERS = [8, 10, 12]
bench_results = []

for workers in BENCH_WORKERS:
    cfg_bench = copy.deepcopy(cfg)
    t_b = cfg_bench.setdefault('train', {})
    a_b = cfg_bench.setdefault('augmentation', {})

    t_b['device'] = 'cuda'
    t_b['batch_size'] = 512
    t_b['num_workers'] = workers
    t_b['prefetch_factor'] = 4
    t_b['max_epochs'] = 1
    t_b['early_stopping_patience'] = 1
    t_b['tables_dir'] = './outputs/tables'
    t_b['checkpoint_dir'] = './outputs/checkpoints'

    # W benchmarku izolujemy wpływ DataLoadera (bez kosztu budowania cache)
    t_b['cache_features'] = 'none'

    a_b['enabled'] = False

    tmp_bench_cfg = f'/tmp/{EXPERIMENT}_bench_w{workers}.yaml'
    with open(tmp_bench_cfg, 'w') as f:
        yaml.dump(cfg_bench, f)

    print(f'\n=== BENCH: workers={workers}, augment=False, batch=512 ===')
    cmd = [sys.executable, 'scripts/train.py', '--config', tmp_bench_cfg, '--seed', str(SEED)]
    t0 = time.perf_counter()
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    wall = time.perf_counter() - t0

    hfiles = sorted(glob.glob('outputs/tables/*_history.json'), key=lambda p: os.path.getmtime(p))
    epoch_time = None
    if hfiles:
        with open(hfiles[-1]) as f:
            d = json.load(f)
        hist = d.get('history', []) if isinstance(d, dict) else d
        if hist and isinstance(hist[0], dict):
            epoch_time = hist[0].get('epoch_time')

    bench_results.append({
        'workers': workers,
        'returncode': proc.returncode,
        'epoch_time_s': epoch_time,
        'wall_time_s': wall,
    })

    print(f'returncode={proc.returncode} | epoch_time={epoch_time} | wall={wall:.1f}s')

print('\n=== PODSUMOWANIE workers ===')
for r in bench_results:
    print(r)

ok = [r for r in bench_results if r['epoch_time_s'] is not None]
if ok:
    best = min(ok, key=lambda r: r['epoch_time_s'])
    print(f"\nNajlepszy num_workers: {best['workers']} (epoch_time={best['epoch_time_s']:.1f}s)")

---
## 8. Trenuj model

Po zakonczeniu trening automatycznie zapisze wyniki na Drive.

In [ ]:
import subprocess

cmd = [sys.executable, 'scripts/train.py', '--config', tmp_cfg, '--seed', str(SEED)]
print('Uruchamiam:', ' '.join(cmd))
print('=' * 70)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('=' * 70)
print(f'Kod wyjscia: {proc.returncode}')

# Natychmiast zapisz na Drive
if proc.returncode == 0:
    drive_path = sync_run_to_drive(EXPERIMENT, SEED)
    print(f'\nDrive: {drive_path}')
else:
    print('\nTrening zakonczyl sie bledem — sprawdz logi powyzej.')

---
## 9. Wizualizacja wynikow

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

files = sorted(glob.glob('outputs/tables/*_history.json'))
assert files, 'Brak history.json'
with open(files[-1]) as f:
    data = json.load(f)

hist    = data['history']
meta    = data.get('meta', {})
test    = data.get('test', {})
best_ep = meta.get('best_epoch')

epochs     = [r['epoch']              for r in hist]
train_loss = [r['train_loss']          for r in hist]
val_loss   = [r['val_loss']            for r in hist]
train_acc  = [r['train_acc']           for r in hist]
val_acc    = [r['val_acc']             for r in hist]
train_f1   = [r.get('train_macro_f1') for r in hist]
val_f1     = [r.get('val_macro_f1')   for r in hist]
lr_curve   = [r.get('lr')             for r in hist]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle(
    f"{meta.get('run_name','run')}  |  "
    f"best val_acc={meta.get('best_val_acc', 0):.4f} @ ep{best_ep}",
    fontsize=12, fontweight='bold')

for ax, y1, y2, ttl, ylim in [
    (axes[0,0], train_loss, val_loss, 'Loss',     None),
    (axes[0,1], train_acc,  val_acc,  'Accuracy', (0, 1.02)),
]:
    ax.plot(epochs, y1, label='train')
    ax.plot(epochs, y2, label='val')
    if best_ep: ax.axvline(best_ep, color='gray', ls='--', alpha=.5)
    ax.set_title(ttl); ax.legend(); ax.grid(alpha=.3)
    if ylim: ax.set_ylim(*ylim)

ax = axes[1,0]
if any(v is not None for v in train_f1):
    ax.plot(epochs, train_f1, label='train')
    ax.plot(epochs, val_f1, label='val')
    ax.set_title('Macro F1'); ax.set_ylim(0,1.02); ax.legend(); ax.grid(alpha=.3)
else: ax.set_visible(False)

ax = axes[1,1]
if any(v is not None for v in lr_curve):
    ax.plot(epochs, lr_curve, color='tab:orange')
    ax.set_title('Learning Rate'); ax.set_yscale('log'); ax.grid(alpha=.3)
else: ax.set_visible(False)

os.makedirs('outputs/figures', exist_ok=True)
plt.tight_layout()
plt.savefig('outputs/figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Wykres zapisany')

In [ ]:
# Per-class metryki + confusion matrix
if test:
    import seaborn as sns
    from src.data.labels import ALL_CLASSES

    print(f"Test Accuracy:     {test['accuracy']:.4f}")
    print(f"Test Macro F1:     {test['macro_f1']:.4f}")
    print(f"Test Macro Prec:   {test['macro_precision']:.4f}")
    print(f"Test Macro Recall: {test['macro_recall']:.4f}")

    cnames = test.get('class_names', ALL_CLASSES)
    f1s = [test.get(f'f1_{c}', 0)        for c in cnames]
    res = [test.get(f'recall_{c}', 0)    for c in cnames]
    prs = [test.get(f'precision_{c}', 0) for c in cnames]

    x = np.arange(len(cnames)); w = .25
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x-w, prs, w, label='Precision')
    ax.bar(x,   res, w, label='Recall')
    ax.bar(x+w, f1s, w, label='F1')
    ax.set_xticks(x); ax.set_xticklabels(cnames, rotation=30, ha='right')
    ax.set_ylim(0, 1.05); ax.set_title('Per-class (test set)')
    ax.legend(); ax.grid(axis='y', alpha=.3)
    plt.tight_layout()
    plt.savefig('outputs/figures/per_class_test.png', dpi=150, bbox_inches='tight')
    plt.show()

    if 'confusion_matrix' in test:
        cm  = np.array(test['confusion_matrix'])
        cmn = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
        fig, ax = plt.subplots(figsize=(11, 9))
        import seaborn as sns
        sns.heatmap(cmn, annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=cnames, yticklabels=cnames, ax=ax, vmin=0, vmax=1)
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        ax.set_title('Confusion Matrix (normalized, test set)')
        plt.tight_layout()
        plt.savefig('outputs/figures/confusion_matrix.png', dpi=150, bbox_inches='tight')
        plt.show()

    # Zaktualizuj wykresy na Drive
    sync_run_to_drive(EXPERIMENT, SEED)
else:
    print('Brak sekcji test w history.')

---
## 10. Trening wielu eksperymentow pod rzad

Każdy model jest zapisywany na Drive od razu po zakonczeniu.  
Jeśli Colab się zresetuje, stracisz tylko bieżący, niedokonczony model.

In [ ]:
EXPERIMENTS_TO_RUN = [
    ('A1', 42),
    ('A2', 42),
    ('A3', 42),
]

for exp, seed in EXPERIMENTS_TO_RUN:
    print(f'\n{"="*70}')
    print(f'EKSPERYMENT: {exp}  seed: {seed}')
    print('='*70)

    cfg2 = load_experiment_config(CONFIG_MAP[exp])
    t2   = cfg2.setdefault('train', {})
    t2['num_workers'] = 4
    t2['device']      = 'cuda'
    t2['batch_size']  = 256
    t2['tables_dir']     = './outputs/tables'
    t2['checkpoint_dir'] = './outputs/checkpoints'

    tmp2 = f'/tmp/{exp}_seed{seed}_colab.yaml'
    with open(tmp2, 'w') as f:
        yaml.dump(cfg2, f)

    proc = subprocess.Popen(
        [sys.executable, 'scripts/train.py', '--config', tmp2, '--seed', str(seed)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end='')
    proc.wait()

    if proc.returncode == 0:
        sync_run_to_drive(exp, seed)
        print(f'Eksperyment {exp} zapisany na Drive.')
    else:
        print(f'BLAD w eksperymencie {exp}!')

---
## 11. Indeks eksperymentow na Drive

In [ ]:
idx_path = f'{DRIVE_RUNS}/index.json'
if os.path.exists(idx_path):
    with open(idx_path) as f:
        idx = json.load(f)
    print(f'{"Run name":<50} {"val_acc":>8} {"test_acc":>9} {"ep":>4}  saved_at')
    print('-' * 90)
    for r in idx:
        val  = f"{r['best_val_acc']:.4f}" if r.get('best_val_acc') else '   -  '
        test = f"{r['test_acc']:.4f}"     if r.get('test_acc')     else '   -  '
        print(f"{r['run_name']:<50} {val:>8} {test:>9} {str(r.get('total_epochs','')):>4}  {r['saved_at']}")
else:
    print('Brak index.json — zadne eksperymenty nie zostaly jeszcze zapisane.')

---
## Wskazowki

**Struktura na Drive po treningu:**
```
Studia/Projekty/DeepLearning/AudioClassification/
├── Dataset/
│   └── SpeechCommands/          ← cache datasetu (raz pobrany)
└── runs/
    ├── index.json               ← tabela wszystkich runow
    ├── A1_CNN_Baseline_seed42/
    │   ├── *_history.json
    │   ├── checkpoints/
    │   │   ├── *_best.pt
    │   │   └── *_last.pt
    │   └── figures/
    │       ├── training_curves.png
    │       ├── per_class_test.png
    │       └── confusion_matrix.png
    └── A2_KWT_Small_seed42/ ...
```

**Czasy (T4, bs=256):** A1 ~2.5 h | A2 ~4 h | A3 ~7 h  
**Limit sesji Colab Free:** ~12 h — A1+A2 mieszcza sie w jednej sesji.